In [ ]:
# ============================================================
# English Listening Video Generator (方案 B — Grouped Multi-Character)
# All-in-one cell: Install → Mount Drive → Clone → Run → Download
# ============================================================

# ===== 1. Install Dependencies =====
!apt-get update -qq && apt-get install -y -qq ffmpeg git
!pip install -q kokoro soundfile torch edge-tts Pillow opencc-python-reimplemented
!pip install -q cn2an pypinyin ordered_set jieba
!apt-get install -y -qq fonts-noto-cjk fonts-dejavu-core
import os; os.makedirs('/content/output', exist_ok=True)
print(f'FFmpeg: {os.popen("ffmpeg -version 2>/dev/null | head -1").read().strip()}')
print('Dependencies installed.')

# ===== 2. Mount Google Drive =====
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/listening_videos'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Google Drive: {DRIVE_DIR}')

# ===== 3. Clone Repository from GitHub =====
REPO_URL = 'https://github.com/collinsgraciano/colab_listening_b.git'
REPO_DIR = '/content/listening_b'
if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}
required = ['mcp_client.py', 'llm_client.py', 'tts_engine.py', 'timeline.py',
            'grouping_b.py', 'video_compose.py', 'pipeline.py', 'topics.json',
            'topic_manager.py']
missing = [f for f in required if not os.path.exists(os.path.join(REPO_DIR, f))]
if missing:
    print(f'Missing files: {missing}')
else:
    print('All scripts cloned successfully!')

# ===== 4. API Keys =====
# MCP OAuth Tokens — array format, one token per line
# Get tokens from: C:\Users\Administrator\.codely-cli\mcp-oauth-tokens.json
# Add/remove tokens freely; pipeline auto-rotates on 积分 errors
MCP_TOKENS = [
    'PASTE_TOKEN_1_HERE',
    'PASTE_TOKEN_2_HERE',
    # 'PASTE_TOKEN_3_HERE',
]

# SenseNova API Key
SENSENOVA_API_KEY = 'PASTE_YOUR_API_KEY_HERE'

# Join tokens for CLI
MCP_TOKENS_STR = ','.join(t.strip() for t in MCP_TOKENS if t.strip() and not t.strip().startswith('PASTE_'))

if not MCP_TOKENS_STR:
    print('\n⚠️  Please paste your MCP tokens above!')
elif SENSENOVA_API_KEY == 'PASTE_YOUR_API_KEY_HERE':
    print('\n⚠️  Please paste your SenseNova API key above!')
else:
    n_tokens = len([t for t in MCP_TOKENS_STR.split(',') if t.strip()])
    os.environ['SENSENOVA_API_KEY'] = SENSENOVA_API_KEY
    print(f'\n✅ MCP Tokens: {n_tokens} token(s) set')
    print(f'✅ SenseNova API Key set ({len(SENSENOVA_API_KEY)} chars)')

    # ===== 5. Run Pipeline =====
    TOPIC = ''                          # e.g. 'Doing Laundry' or '' for random
    CEFR = 'A2'                         # A1, A2, B1, B2, C1, C2
    NUM_LINES = 18                      # Number of dialogue lines
    STRUCTURE = 'original'             # 'original' or 'enhanced'
    OUTPUT_DIR = DRIVE_DIR             # Pipeline creates subfolder per video
    RESUME = True                      # Resume incomplete run if exists; else new topic

    cmd = f'cd /content/listening_b && python pipeline.py '
    if TOPIC.strip():
        cmd += f'--topic "{TOPIC}" '
    cmd += f'--cefr {CEFR} --num-lines {NUM_LINES} --structure {STRUCTURE} '
    cmd += f'--output "{OUTPUT_DIR}" '
    cmd += f'--mcp-tokens {MCP_TOKENS_STR} --api-key {SENSENOVA_API_KEY}'
    if RESUME:
        cmd += ' --resume'

    print('\nRunning:')
    print(cmd)
    print()
    !{cmd}

    # ===== 6. Find run directory (YouTube-title-named subfolder) =====
    import json
    run_dir = None
    for sub in os.listdir(OUTPUT_DIR):
        sub_path = os.path.join(OUTPUT_DIR, sub)
        if os.path.isdir(sub_path) and os.path.exists(os.path.join(sub_path, 'script.json')):
            # Completed runs have no checkpoint.json
            if not os.path.exists(os.path.join(sub_path, 'checkpoint.json')):
                run_dir = sub_path
                break
    if not run_dir:
        # Fallback: most recent subfolder with script.json
        subs = [(os.path.getmtime(os.path.join(OUTPUT_DIR, s)), s) for s in os.listdir(OUTPUT_DIR)
                if os.path.isdir(os.path.join(OUTPUT_DIR, s)) and
                os.path.exists(os.path.join(OUTPUT_DIR, s, 'script.json'))]
        if subs:
            subs.sort(reverse=True)
            run_dir = os.path.join(OUTPUT_DIR, subs[0][1])
    if run_dir:
        print(f'\n📁 Run directory: {run_dir}')
        script_data = json.load(open(f'{run_dir}/script.json', encoding='utf-8'))
        print(f'   Title: {script_data.get("youtube_title", "")}')
        # Find video file (named by YouTube title, not final_video.mp4)
        vid_dir = os.path.join(run_dir, 'videos')
        vid_p = None
        if os.path.isdir(vid_dir):
            mp4s = [os.path.join(vid_dir, f) for f in os.listdir(vid_dir) if f.endswith('.mp4') and '_4K' not in f]
            if mp4s:
                vid_p = max(mp4s, key=os.path.getsize)
            # Also find 4K version
            mp4s_4k = [os.path.join(vid_dir, f) for f in os.listdir(vid_dir) if f.endswith('_4K.mp4')]
            vid_4k_p = max(mp4s_4k, key=os.path.getsize) if mp4s_4k else None
        else:
            vid_4k_p = None
        if vid_p and os.path.exists(vid_p):
            print(f'   Video: {os.path.basename(vid_p)} ({os.path.getsize(vid_p)/(1024*1024):.1f}MB)')
        if vid_4k_p and os.path.exists(vid_4k_p):
            print(f'   4K Video: {os.path.basename(vid_4k_p)} ({os.path.getsize(vid_4k_p)/(1024*1024):.1f}MB)')
    else:
        print('⚠️ Could not find run directory.')
        run_dir = OUTPUT_DIR
        vid_p = None
        vid_4k_p = None

    # ===== 7. Download Results =====
    from google.colab import files

    for label, path in [
        ('Video', vid_p),
        ('4K Video', vid_4k_p),
        ('Thumbnail', f'{run_dir}/thumbnail.jpg'),
        ('Script', f'{run_dir}/script.json'),
        ('YouTube metadata', f'{run_dir}/youtube_metadata.json'),
    ]:
        if path and os.path.exists(path):
            size = os.path.getsize(path)
            size_str = f'{size/(1024*1024):.1f}MB' if size > 1024*1024 else f'{size//1024}KB'
            print(f'{label}: {path} ({size_str})')
            files.download(path)
        else:
            print(f'{label}: not found')

    # ===== 8. Preview Video (optional) =====
    from IPython.display import HTML
    from base64 import b64encode

    video_path = vid_p
    if video_path and os.path.exists(video_path):
        with open(video_path, 'rb') as f:
            video_b64 = b64encode(f.read()).decode()
        html = f'''
        <video width="640" height="360" controls>
            <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        </video>
        '''
        display(HTML(html))
    else:
        print('Video not found for preview.')
    print('\n✅ All done!')